# GLD ML Prediction Horizon Scan — Research Report
**Date:** 2026-05-10 ~ 2026-05-11  
**Researcher:** Joon Jeremy Chun  
**Location:** `research/horizon_scan_gld/`  

---

## 1. Background and Motivation

### Overall System Architecture (Layer 0~3)

| Layer | Role | Status |
|-------|------|--------|
| **0** | Human macro judgment — which assets to include | Active |
| **1** | Strategy optimization — grid search over 4 strategy families | Active |
| **2** | ML prediction — OLS/Ridge/Lasso/ElasticNet predicts N-day forward return → position weight | Active (130d fixed) |
| **3** | SINDy nonlinear combination — return maximization | Not yet implemented |

### The Fixed Parameter in the Current System
Layer 2 currently uses a **fixed 130-day prediction target**.  
**Key question:** Is 130 days optimal? Or is some other N-day horizon better?

---

## 2. Experiment Design

### Idea: Dense Window + Horizon Scan

**Current system:**
```
Training windows: 1m, 3m, 6m, 1y (sparse intervals)
Prediction target: 130 days (fixed)
```

**This experiment:**
```
Training windows: 20d ~ 520d (5-day intervals, 101 windows) ← Strategy 1 only, newly computed
Prediction target: 5d ~ 500d (5-day intervals, 100 values)  ← full scan
→ Determine the optimal N-day target from data
```

### Feature Construction (~1,141 features total)

| Strategy | # Features | Data Source |
|----------|------------|-------------|
| **S1 Adaptive Band (dense)** | 101 windows × topN | Newly computed |
| **S2 MA Crossover** | Existing horizons × topN | Reused from existing anchors |
| **S3 Adaptive Vol Band** | Existing horizons × topN | Reused from existing anchors |

Each feature = **daily position signal (0/1)** produced by applying the corresponding parameters to the selection period price data.

### ML Models (4 types)
- OLS (ordinary least squares, no regularization)
- Ridge (L2 regularization, 30-alpha CV)
- Lasso (L1 regularization, 30-alpha CV, automatic feature selection)
- ElasticNet (L1+L2 mix, 30-alpha × 5 l1_ratio CV)

**Selection criterion:** Minimum CV-MSE (TimeSeriesSplit 5-fold)

### Experiment Scale
- Anchors: 10 (2021-05 ~ 2025-11, every 6 months)
- Horizon scan: 100 values (5~500 days)
- top10 experiment: 10 × 100 = 1,000 trials (72 minutes)
- top20 experiment: 10 × 100 = 1,000 trials (82 minutes)
- **Asset: GLD only** (other assets will adopt the same optimal N-day)

### Independence Principle
- Folder: `research/horizon_scan_gld/` — fully isolated from the main pipeline
- No modifications to original codebase
- Data files referenced read-only from original locations

---

## 3. Result Visualization

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load results
BASE = '../../research/horizon_scan_gld/outputs/'
t10 = pd.read_csv(BASE + 'horizon_scan_top10_summary.csv').sort_values('horizon_days')
t20 = pd.read_csv(BASE + 'horizon_scan_top20_summary.csv').sort_values('horizon_days')

print(f'top10: {len(t10)} horizons')
print(f'top20: {len(t20)} horizons')
print()
print('=== top10 Top5 (by Forward Return) ===')
print(t10.nlargest(5,'avg_forward_return')[['horizon_days','avg_forward_return','avg_cv_mse','dir_accuracy','best_model_mode']].to_string(index=False))
print()
print('=== top20 Top5 (by Forward Return) ===')
print(t20.nlargest(5,'avg_forward_return')[['horizon_days','avg_forward_return','avg_cv_mse','dir_accuracy','best_model_mode']].to_string(index=False))

In [ ]:
# ============================================================
# Graph 1 & 2: top10
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Top-10 Features  |  4 Models (OLS/Ridge/Lasso/ElasticNet)', fontsize=13, fontweight='bold')

# Graph 1: MSE
ax = axes[0]
ax.plot(t10['horizon_days'], t10['avg_cv_mse']*1000, 'b-o', ms=3, lw=1.5)
ax.axvline(130, color='green', ls='--', lw=1.5, label='Current system (130d)')
best_mse = t10.nsmallest(1,'avg_cv_mse').iloc[0]
ax.axvline(best_mse.horizon_days, color='orange', ls=':', lw=1.5, label=f'MSE min ({int(best_mse.horizon_days)}d)')
ax.set_xlabel('Prediction Horizon (days)')
ax.set_ylabel('CV-MSE (x1000) — lower is better')
ax.set_title('Graph 1: Prediction Error (top10)')
ax.legend()
ax.grid(alpha=0.3)

# Graph 2: Return
ax = axes[1]
ax.plot(t10['horizon_days'], t10['avg_forward_return']*100, 'b-o', ms=3, lw=1.5)
ax.axhline(0, color='black', lw=0.8)
ax.axvline(130, color='green', ls='--', lw=1.5, label='Current system (130d)')
best_ret = t10.nlargest(1,'avg_forward_return').iloc[0]
ax.axvline(best_ret.horizon_days, color='red', ls=':', lw=1.5, label=f'Return peak ({int(best_ret.horizon_days)}d)')
ax.set_xlabel('Prediction Horizon (days)')
ax.set_ylabel('Avg Forward Return (%)')
ax.set_title('Graph 2: Forward Return (top10)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Graph 3 & 4: top20
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Top-20 Features  |  4 Models (OLS/Ridge/Lasso/ElasticNet)', fontsize=13, fontweight='bold')

# Graph 3: MSE
ax = axes[0]
ax.plot(t20['horizon_days'], t20['avg_cv_mse']*1000, 'r-o', ms=3, lw=1.5)
ax.axvline(130, color='green', ls='--', lw=1.5, label='Current system (130d)')
best_mse = t20.nsmallest(1,'avg_cv_mse').iloc[0]
ax.axvline(best_mse.horizon_days, color='orange', ls=':', lw=1.5, label=f'MSE min ({int(best_mse.horizon_days)}d)')
ax.set_xlabel('Prediction Horizon (days)')
ax.set_ylabel('CV-MSE (x1000) — lower is better')
ax.set_title('Graph 3: Prediction Error (top20)')
ax.legend()
ax.grid(alpha=0.3)

# Graph 4: Return
ax = axes[1]
ax.plot(t20['horizon_days'], t20['avg_forward_return']*100, 'r-o', ms=3, lw=1.5)
ax.axhline(0, color='black', lw=0.8)
ax.axvline(130, color='green', ls='--', lw=1.5, label='Current system (130d)')
best_ret = t20.nlargest(1,'avg_forward_return').iloc[0]
ax.axvline(best_ret.horizon_days, color='red', ls=':', lw=1.5, label=f'Return peak ({int(best_ret.horizon_days)}d)')
ax.set_xlabel('Prediction Horizon (days)')
ax.set_ylabel('Avg Forward Return (%)')
ax.set_title('Graph 4: Forward Return (top20)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Comparison: top10 vs top20 side by side
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('top10 vs top20 Comparison', fontsize=13, fontweight='bold')

# MSE comparison
ax = axes[0]
ax.plot(t10['horizon_days'], t10['avg_cv_mse']*1000, 'b-o', ms=3, lw=1.5, label='top10')
ax.plot(t20['horizon_days'], t20['avg_cv_mse']*1000, 'r-s', ms=3, lw=1.5, label='top20', alpha=0.8)
ax.axvline(130, color='green', ls='--', lw=1.5, label='Current 130d')
ax.set_xlabel('Prediction Horizon (days)')
ax.set_ylabel('CV-MSE (x1000)')
ax.set_title('MSE: top10 vs top20')
ax.legend()
ax.grid(alpha=0.3)

# Return comparison
ax = axes[1]
ax.plot(t10['horizon_days'], t10['avg_forward_return']*100, 'b-o', ms=3, lw=1.5, label='top10')
ax.plot(t20['horizon_days'], t20['avg_forward_return']*100, 'r-s', ms=3, lw=1.5, label='top20', alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.axvline(130, color='green', ls='--', lw=1.5, label='Current 130d')
ax.axvline(145, color='purple', ls=':', lw=1.5, label='Optimal 145d')
ax.set_xlabel('Prediction Horizon (days)')
ax.set_ylabel('Avg Forward Return (%)')
ax.set_title('Return: top10 vs top20')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## 4. Results and Analysis

### Numerical Summary

| Metric | top10 | top20 |
|--------|-------|-------|
| **Best Return horizon** | **145 days** (+0.93%) | **145 days** (+0.86%) |
| **2nd best** | 150 days (+0.77%) | 150 days (+0.84%) |
| **Lowest MSE horizon** | 155 days (0.000822) | 155 days (0.000847) |
| **Best Dir Accuracy** | 35d (80%), 60d (80%) | 155d (71%) |
| **Current system 130d** | Rank 10 (+0.46%) | Outside top 10 |
| **Dominant model** | Lasso / ElasticNet | ElasticNet |

### Pattern Interpretation

**Graph 1 & 3 (MSE — Prediction Error):**
- **5~20 days**: Low MSE — naturally low because short-term volatility is small (but returns are negative)
- **50~130 days**: MSE spikes — hardest region to predict
- **130~160 days**: MSE drops sharply — prediction accuracy recovers in this zone
- **160+ days**: MSE rises again irregularly

**Graph 2 & 4 (Forward Return):**
- **35~40 days**: Short-term sweet spot (80% directional accuracy)
- **50~120 days**: Generally low or negative returns
- **140~155 days**: **Consistent peak across both top10 and top20** ← key finding
- **160+ days**: Sharp drop

### Key Conclusion

> **Optimal N = 145 days (by return) / 155 days (by MSE)**  
> 
> The current system's 130-day target is reasonable but not optimal.  
> **145~155 days is consistently better** across both top10 and top20 — a robust finding.  
> ElasticNet dominates with larger feature sets (top20), indicating L1+L2 regularization is preferred when dimensionality increases.

---

## 5. Next Steps

1. **Immediate**: Test adjusting `target_horizon_days=130` → `145` in the live system
2. **Medium term**: Run the same experiment for other assets (BRK-B, QQQ, RKLB) — do they share the same optimal N?
3. **Layer 3**: SINDy nonlinear combination of prediction signals → return maximization (separate future work)

---

## 6. File Reference

```
research/horizon_scan_gld/
  strategy1_dense_windows.py      # Strategy1 dense optimization (modified from original script 11)
  run_strategy1_dense.py          # Runner for all 10 anchors sequentially
  run_horizon_scan.py             # ML horizon scan master (supports --top-n argument)
  run_both_scans.py               # Sequential runner: top10 then top20 automatically
  outputs/
    strategy1_dense/anchor_*/     # Per-anchor dense optimization results
    horizon_scan_top10_summary.csv
    horizon_scan_top20_summary.csv
    horizon_scan_graphs.png
```